In [ ]:
import pandas as pd
import numpy as npp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import joblib


print(f'Rebuilding data pipeline...')


AGG_RULES = {
    'OIL_RATE_NORM':         'mean',
    'AVG_WHP_P':             'mean',
    'AVG_DOWNHOLE_PRESSURE': 'mean',
    'WCT':                   'mean',
    'GOR':                   'median',
    'DRAWDOWN':              'mean',
    'DAYS_ON_PROD':          'max',
    'ON_STREAM_HRS':         'sum',
}

df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()
df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(
    lambda x : x.clip(upper= x.quantile(0.99))
)

TARGET_WELLS = ['15/9-F-12', '15/9-F-14']
monthly_well_data = {}
for well_name,well_df in df.groupby('NPD_WELL_BORE_NAME'):
    if(well_name not in TARGET_WELLS):
        print(f'this well has no clean data')
        continue
    cols_availble = {k: v for k, v in AGG_RULES.items() if k in well_df.columns}
    monthly = well_df[list(cols_availble.keys())].resample('ME').agg(
        cols_availble
    ).dropna(subset=['OIL_RATE_NORM'])
    monthly_well_data[well_name] = monthly


 
def engineer_features(month_df):
    df_feature = month_df.copy()
    df_feature['lag_1_oil'] = df_feature['OIL_RATE_NORM'].shift(1)
    df_feature['lag_3_oil'] = df_feature['OIL_RATE_NORM'].shift(3)
    df_feature['lag_6_oil'] = df_feature['OIL_RATE_NORM'].shift(6)
    df_feature['lag_1_wct'] = df_feature['WCT'].shift(1) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_3_wct'] = df_feature['WCT'].shift(3) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_6_wct'] = df_feature['WCT'].shift(6) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_1_whp'] = df_feature['AVG_WHP_P'].shift(1) if 'AVG_WHP_P' in df_feature.columns else np.nan
    df_feature['rolling_3m'] = df_feature['OIL_RATE_NORM'].rolling(window=3, min_periods=2).mean()
    df_feature['rolling_6m'] = df_feature['OIL_RATE_NORM'].rolling(window=6, min_periods=3).mean()
    df_feature['month_over_month'] = df_feature['OIL_RATE_NORM'].pct_change().clip(-1, 1)
    df_feature['rate_vs_6m_avg'] = df_feature['lag_1_oil'] / (df_feature['rolling_6m'] + 1e-6)
    df_feature['wct_change'] = df_feature['WCT'].diff()
    df_feature['well_age'] = df_feature['DAYS_ON_PROD'] / 365
    return df_feature